In [67]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import datetime
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import random

from transformers import BertTokenizer
from transformers import BertForSequenceClassification

from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.utils import class_weight
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

def set_seed(seed):
    """Sets the seed for reproducibility."""
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU.
    np.random.seed(seed)  # Numpy module.
    random.seed(seed)  # Python's random module.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.enabled = False

# Example usage:
seed = 1000
set_seed(seed)

# Load and Preprocess Data

In [74]:
df = pd.read_csv('./data/wyoming.csv')
df = df[['rating','text']]
df_train, df_test = train_test_split(df, test_size=0.2, stratify=df['rating'])

In [ ]:
# df_train = pd.read_csv('../duncan/data/hotdog-train.csv')
# df_test = pd.read_csv('../duncan/data/hotdog-test.csv')
# df_train.head()

,rating,text
0,5,"This is one of those ""hole in the wall"" places..."
1,5,💘 it
2,5,Best place in town for hotdogs and root beer
3,5,Food is so good
4,5,Always a hit!


In [75]:
def preprocess_text(text):
    text = str(text)
    text = text.replace('(Translated by Google)', ' ')
    text = text.replace('\n', ' ')
    text = text.lower() # for uncased
    return text

df_train['text'] = df_train['text'].apply(preprocess_text)
df_test['text'] = df_test['text'].apply(preprocess_text)

df_train['rating'] = df_train['rating'] - 1
df_test['rating'] = df_test['rating'] - 1

In [76]:
print('Train set sample')
print(df_train.sample(3))
print()

print('Test set sample')
print(df_test.sample(3))

Train set sample
        rating                                               text
137681       4                                               good
211436       3  the food was great but the seating was really ...
81781        4  very nice. nice sitting areas in the grand sta...

Test set sample
        rating                                               text
149747       0  i didn't enjoy it. the staff was cool. but foo...
202208       4                                 great meat prices!
14073        4  i had some sewing done. excellent work, very r...


In [77]:
X_train = df_train['text'].values.tolist()
y_train = df_train['rating'].values.tolist()
X_test = df_test['text'].values.tolist()
y_test = df_test['rating'].values.tolist()

Set hardware configuration

In [78]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

device

device(type='mps')

Load BERT tokenizer and model

In [93]:
model_name = "bert-base-uncased" 
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=5).to(device)
print(model)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

Predict individual reviews on untrained model

In [82]:
def tokenize_text(text):
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    return inputs

def predict_sentiment(text):
    inputs = tokenize_text(text)
    inputs = {key: val.to(device) for key, val in inputs.items()} #move the inputs to the correct device.
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probabilities = torch.softmax(logits, dim=1)
        predicted_class = torch.argmax(probabilities, dim=1).item()
    return predicted_class

# Simple test cycle of the untrained model
review = 'The best chicken ever'
label = predict_sentiment(review)
# meaning = ['negative', 'neutral/average', 'good']
meaning = ['1 star', '2 stars', '3 stars', '4 stars', '5 stars']

print('Review: ', review)
print(f'Predicted rating: {label}, {meaning[label]}')

Review:  The best chicken ever
Predicted rating: 2, 3 stars


# BERT Model

## Model Definition

In [94]:
class BertBaseUncased(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        inputs = self.tokenizer(
            text,
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            'input_ids': inputs['input_ids'].flatten(),
            'attention_mask': inputs['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

Freezing parameters (except for classfication layer)

In [ ]:
# for param in model.bert.parameters():
#     param.requires_grad = False

Check trainble model parameters

In [95]:
def count_trainable_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Trainable parameters: {count_trainable_params(model):,}")

Trainable parameters: 109,486,085


Define training parameters

In [96]:
# Create rating class weights
def calculate_class_weights(labels):
    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(labels), y=labels)
    return torch.tensor(class_weights, dtype=torch.float)

batch_size = 16 # Keep small, below 32
max_length = 128
learning_rate = 2e-5
weight_decay = .01

train_dataset = BertBaseUncased(X_train, y_train, tokenizer, max_length=max_length)
test_dataset = BertBaseUncased(X_test, y_test, tokenizer, max_length=max_length)

train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Calculate class weights only after the split
class_weights = calculate_class_weights(y_train).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

print('Class weights:', class_weights)

Class weights: tensor([3.8991, 5.1965, 2.1201, 0.9059, 0.3361], device='mps:0')


Plotting function

In [97]:
def plot_lists_with_colors(list1, list2, label1="List 1", label2="List 2", color1="blue", color2="red"):
    # Ensure lists have equal length (or adjust as needed)
    min_len = min(len(list1), len(list2))
    x_values = np.arange(min_len)

    plt.figure(figsize=(10, 6))  # Adjust figure size if needed
    plt.plot(x_values, list1[:min_len], color=color1, label=label1, linestyle='none', marker='o')
    plt.plot(x_values, list2[:min_len], color=color2, label=label2, linestyle='none', marker='o')

    current_time = datetime.datetime.now()
    
    plt.xlabel('Epoch') 
    plt.ylabel('Accuracy (%)')
    plt.title(f'Bert - Midwest data ({current_time.strftime("%Y-%M-%d %H:%M")})')

    plt.legend()
    plt.tight_layout()
    plt.grid(True, linestyle='--', alpha=0.7)
    
    file_name = f'./graphs_bert/model-{current_time.strftime("%Y-%M-%d--%H%M")}.png'
    plt.savefig(file_name)
    
    plt.show()

In [ ]:
epochs = 5
accs = [[], []]  # 0 = Train, 1 = Test
best_test_acc = 0

for epoch in range(epochs):
    # ---- Training ---- #
    model.train()
    train_loss = 0
    all_preds, all_labels = [], []

    for batch in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}/{epochs}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

        loss = outputs.loss
        train_loss += loss.item()

        loss.backward()
        optimizer.step()

        _, preds = torch.max(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_train_loss = train_loss / len(train_dataloader)
    train_acc = accuracy_score(all_labels, all_preds) * 100
    accs[0].append(train_acc)

    # ---- Evaluation ---- #
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)
            total_loss += loss.item()

            _, preds = torch.max(outputs.logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_test_loss = total_loss / len(test_dataloader)
    test_acc = accuracy_score(all_labels, all_preds) * 100
    accs[1].append(test_acc)

    print(f"Epoch {epoch + 1}/{epochs}")
    print(f"  Train loss: {avg_train_loss:.4f}, accuracy: {train_acc:.2f}%")
    print(f"  Test  loss: {avg_test_loss:.4f}, accuracy: {test_acc:.2f}%")
    print("  Classification Report:\n")
    print(classification_report(all_labels, all_preds, digits=3))

    # ---- Save Best Model ---- #
    if test_acc > best_test_acc:
        best_test_acc = test_acc
        torch.save(model.state_dict(), f'saved_models/best_model_epoch{epoch + 1}.pt') # update path
        print(f"  ✅ Saved new best model (epoch {epoch + 1})")

# ---- Plot Accuracy ---- #
plot_lists_with_colors(accs[0], accs[1], label1="Train Accuracy", label2="Test Accuracy")


Save the model

In [89]:
torch.save(model.state_dict(), f'./saved_models/bert_frozen.pth')

## Testing Saved Models

In [91]:
model = BertForSequenceClassification.from_pretrained(model_name, num_labels=5)
model.load_state_dict(torch.load('./saved_models/bert_frozen.pth'))
model.to(device)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [92]:
model.eval()
total_loss = 0
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in test_dataloader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        loss = criterion(outputs.logits, labels)
        total_loss += loss.item()

        _, preds = torch.max(outputs.logits, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

avg_test_loss = total_loss / len(test_dataloader)
test_acc = accuracy_score(all_labels, all_preds) * 100
accs[1].append(test_acc)

print(f"Epoch {epoch + 1}/{epochs}")
print(f"  Train loss: {avg_train_loss:.4f}, accuracy: {train_acc:.2f}%")
print(f"  Test  loss: {avg_test_loss:.4f}, accuracy: {test_acc:.2f}%")
print("  Classification Report:\n")
print(classification_report(all_labels, all_preds, digits=3))

Epoch 5/5
  Train loss: 0.9965, accuracy: 60.39%
  Test  loss: 1.7248, accuracy: 60.16%
  Classification Report:

              precision    recall  f1-score   support

           0      0.543     0.079     0.137      2407
           1      0.000     0.000     0.000      1806
           2      0.349     0.007     0.013      4428
           3      0.329     0.019     0.036     10361
           4      0.606     0.996     0.754     27929

    accuracy                          0.602     46931
   macro avg      0.366     0.220     0.188     46931
weighted avg      0.494     0.602     0.465     46931



/Users/andy/anaconda3/envs/erdos/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/andy/anaconda3/envs/erdos/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/Users/andy/anaconda3/envs/erdos/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

## Suggestions

 Set aside cross-validation set - 
 
 Total random seed pytorch level torch.manual_seed() - DONE
  
 Early exit with high number of epochs
 
 Play with groupings of ratings (1--5, pos/neg only -- create neutral based on logits) - TRIED WITH 0--4

 Learning rates: 5e-5, 4e-5, 3e-5, and 2e-5, .0001. Learning rate scheduler

 Investigate BERT classifier more

 Try different BERT model -- roberta? https://huggingface.co/AnkitAI/reviews-roberta-base-sentiment-analysis